# QAM: Modulating & Demodulating a Complex-Valued Signal

*Python/Colab conversion of the MATLAB script `qamdemod.m`*
(from **Software Receiver Design**, Johnson, Sethares & Klein).

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/oalnaseri/CommSystems_Course/blob/main/06_qamdemod.ipynb)

> **How to run:** Click the **Open in Colab** badge above, then run every cell top-to-bottom with `Shift + Enter`, or use **Runtime -> Run all**. No login needed — just click and run.

**Quadrature Amplitude Modulation (QAM)** carries *two* independent messages at once on a single carrier — one on a cosine (the **in-phase / real** part) and one on a sine (the **quadrature / imaginary** part). This notebook builds a **complex-valued** message, modulates it up to a 1000 Hz carrier, then demodulates and low-pass filters it to recover both parts.

A key point: the demodulator's phase (`ph`) is set to **match** the modulator's phase (`th`), so the complex signal comes back cleanly. (Try mismatching them at the end to see what happens!)

No installation needed — `numpy`, `scipy`, `matplotlib` and `plotly` are pre-installed in Colab.

## 1. Setup

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import remez, lfilter          # remez = MATLAB firpm; lfilter = MATLAB filter
from scipy.signal.windows import hamming          # MATLAB hamming(M)

try:
    import plotly.graph_objects as go
    from plotly.subplots import make_subplots
except ModuleNotFoundError:
    import subprocess, sys
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'plotly'])
    import plotly.graph_objects as go
    from plotly.subplots import make_subplots

%matplotlib inline
plt.rcParams['figure.figsize'] = (9, 4)

## 2. Textbook helper functions (re-implemented in Python)

`qamdemod.m` uses two toolbox helpers from *Software Receiver Design*:

| MATLAB helper | Purpose | Python here |
|---|---|---|
| `pam(N, M, Var)` | random length-`N` M-ary PAM sequence with variance `Var` | `pam` |
| `plotspec(x, Ts)` | waveform + magnitude spectrum (handles complex signals) | `plotspec` |

Since QAM is complex, `plotspec` here also plots the spectrum of complex inputs (the spectrum is *not* symmetric — that's the whole point of QAM).

In [ ]:
def pam(N, Mlev, Var):
    """Random length-N M-ary PAM sequence with variance Var (port of pam.m).

    MATLAB: seq = (2*floor(M*rand(1,N)) - M + 1) * sqrt(3*Var/(M^2-1))
    For Mlev=2, Var=1 the alphabet is {-1, +1}.
    """
    return (2*np.floor(Mlev*np.random.rand(N)) - Mlev + 1) * np.sqrt(3*Var/(Mlev**2 - 1))


def plotspec(x, Ts, flim=None, title=''):
    """Waveform + magnitude spectrum (Python port of plotspec.m), optional zoom.

    Works for complex x: the real part of the waveform is plotted, and the FFT
    magnitude (which is generally NOT symmetric for complex signals) is shown.
    """
    x = np.asarray(x); N = len(x)
    t = Ts * np.arange(1, N + 1)
    ssf = np.arange(-N/2, N/2) / (Ts * N)
    fxs = np.fft.fftshift(np.fft.fft(x))
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(9, 6))
    ax1.plot(t, np.real(x))
    ax1.set_xlabel('seconds'); ax1.set_ylabel('amplitude (real part)')
    ax1.set_title(title or 'Waveform'); ax1.grid(True)
    ax2.plot(ssf, np.abs(fxs))
    ax2.set_xlabel('frequency (Hz)'); ax2.set_ylabel('magnitude')
    ax2.set_title('Magnitude spectrum'); ax2.grid(True)
    if flim is not None:
        lo, hi = (-flim, flim) if np.isscalar(flim) else (flim[0], flim[1])
        ax2.set_xlim(lo, hi)
    fig.tight_layout(); plt.show()

## 3. Build the complex QAM message and modulate

Two independent binary PAM streams form the real and imaginary parts of the complex message `m`. After pulse shaping, multiplying by `exp(j(2π f_c t + θ))` and taking the **real part** produces the actual transmitted (real-valued) passband signal `v`.

In [ ]:
np.random.seed(0)                     # reproducible random symbols

N = 10000        # number of symbols
M = 20           # oversampling factor
Ts = 0.0001      # sampling interval
j = 1j

time = Ts * (N*M - 1)
t = np.arange(0, time + Ts/2, Ts)     # time vector of length N*M

m = pam(N, 2, 1) + j*pam(N, 2, 1)     # complex message (real + j*imag)
ps = hamming(M)                       # pulse shape of width M

fc = 1000; th = -1.0                  # carrier frequency and phase
mup = np.zeros(N*M, dtype=complex); mup[::M] = m   # oversample by M
mp = lfilter(ps, 1, mup)              # convolve pulse shape with data

v = np.real(mp * np.exp(j*(2*np.pi*fc*t + th)))    # transmitted passband signal

print(f'{N} complex symbols, waveform length = {len(v)} samples')

## 4. Demodulate and low-pass filter

Multiplying by `exp(-j(2π f_0 t + φ))` mixes the signal back down to baseband. Because the demod frequency `f0` and phase `ph` **match** the transmitter's `fc` and `th`, the low-pass filter output `s` reconstructs the original complex message (scaled by ½, hence the `2*s` later).

In [ ]:
f0 = 1000; ph = -1.0                  # demod frequency and phase (matched to tx)
x = v * np.exp(-j*(2*np.pi*f0*t + ph))    # demodulate v

l = 50                                # filter order
f = [0, 0.2, 0.25, 1]                 # band edges (normalised to Nyquist = 1)
a = [1, 0]                            # pass then stop
b = remez(l + 1, f, a, fs=2)          # design LPF (firpm equivalent)
s = lfilter(b, 1, x)                  # s = LPF{x}
print('demodulated + filtered signal s computed:', s.dtype)

## 5. Spectra at each stage  

Watch the message move up to the carrier and back:
* **`mp`** — baseband complex message (energy near 0 Hz; note the spectrum is *asymmetric* — a hallmark of a complex signal).
* **`v`**  — transmitted passband (energy shifted up to ±1000 Hz).
* **`x`**  — demodulated (energy back near baseband, plus an image up near 2000 Hz).
* **`s`**  — after the low-pass filter (clean baseband message recovered).

In [ ]:
plotspec(mp, 1/M, title='(1) baseband message mp')

In [ ]:
plotspec(v, 1/M, title='(2) transmitted signal v')

In [ ]:
plotspec(x, 1/M, title='(3) demodulated signal x')

In [ ]:
plotspec(s, 1/M, title='(4) demodulated signal after LPF s')

## 6. Transmitted vs. recovered — real part  

Overlaying the original real part of `mp` with the recovered `2*s` (the factor of 2 undoes the ½ loss from demodulation). They should sit almost on top of each other.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(np.real(mp), label='transmitted real(mp)')
ax.plot(np.real(2*s), 'r', label='recovered real(2*s)', alpha=0.75)
ax.set_xlim(0, 1000)                 # zoom in so individual pulses are visible
ax.set_xlabel('sample'); ax.set_ylabel('amplitude')
ax.set_title('Real part: transmitted vs. recovered'); ax.legend(); ax.grid(True)
fig.tight_layout(); plt.show()

## 7. Transmitted vs. recovered — imaginary part 

The imaginary (quadrature) part, over a short window. The recovered signal is shifted by the filter delay `l/2`, so we offset its index by that amount to line the two curves up.

In [ ]:
half = l // 2                        # filter group delay = l/2 samples

# MATLAB indices 500:1000 are 1-indexed; use 499:1000 in Python
tx_imag  = np.imag(mp[499:1000])
rec_imag = np.imag(2*s[499 + half : 1000 + half])

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(tx_imag, label='transmitted imag(mp)')
ax.plot(rec_imag, 'r', label='recovered imag(2*s), delay-aligned', alpha=0.75)
ax.set_xlabel('sample (within window)'); ax.set_ylabel('amplitude')
ax.set_title('Imaginary part: transmitted vs. recovered'); ax.legend(); ax.grid(True)
fig.tight_layout(); plt.show()

corr = np.corrcoef(tx_imag, rec_imag)[0, 1]
print(f'correlation (imag, delay-aligned): {corr:.4f}')

## 8. Notes & things to try

- **Why QAM matters:** it doubles data rate by sending two independent PAM streams on the same carrier — one 'in-phase' (cosine), one 'quadrature' (sine). Wi-Fi, LTE, cable modems and digital TV all use QAM.
- **Complex spectra are asymmetric:** unlike a real signal, the FFT magnitude of the complex message `mp` is *not* mirror-symmetric — that asymmetry is exactly what lets QAM pack two messages in.
- **Phase matters!** Here `th = ph = -1.0`, so the demodulator is perfectly phase-locked. **Try mismatching them** — set `ph = -1.0 + 0.5` and re-run: the real and imaginary parts start to **mix into each other** (a rotation of the constellation). Correcting this is the job of a *carrier-recovery / phase-locked loop* in a real receiver.
- **The factor of 2:** demodulation halves the amplitude, so we compare against `2*s`.
- **Filter delay:** the LPF adds a group delay of `l/2` samples; that's why the imaginary-part comparison offsets the recovered signal by `l/2`.
- **firpm/filter mapping:** `firpm(l,f,a)` → `remez(l+1, f, a, fs=2)`; `filter(b,1,x)` → `lfilter(b,1,x)`; MATLAB 1-indexing → subtract 1 in Python.
